# Hybrid Rocket Motor — Flight Thrust Reconstruction & Trajectory Simulation

## Full Data Analysis Pipeline

This notebook performs a complete, step-by-step scientific analysis to reconstruct the flight thrust curve of a hybrid rocket motor from onboard pressure telemetry, using ground-test calibration data from four independent test firings.

### Objectives
1. **Load and characterize** all available ground-test and flight datasets
2. **Cross-compare** hot-fire tests (HFT1, HFT3, HFT4) and the static motor test (SMT0033)
3. **Calibrate** a chamber-pressure-to-thrust transfer function using load-cell-equipped tests
4. **Reconstruct** the flight thrust curve using a per-phase approach:
   - *Phase 1 (ignition transient):* Shape from HFT3/HFT4, scaled to flight Pc
   - *Phase 2 (steady burn):* Calibrated Pc → Thrust model
   - *Phase 3 (tail-off):* Blowdown-matched to SMT0033
5. **Validate** the reconstruction against independent metrics
6. **Export** a `.eng` thrust file and run a **RocketPy** trajectory simulation
7. **Compare** simulated trajectory with flight telemetry (Fluctus altimeter)

### Data Sources

| Dataset | Sample Rate | Sensors | Ptank₀ | Purpose |
|---------|-----------|---------|--------|---------|
| **HFT1** | 100 Hz | Pc, Ptank, Thrust (LC) | ~56 bar | Calibration (high Pc peak) |
| **HFT3** | 100 Hz | Pc, Ptank, Thrust (LC) | ~51 bar | Calibration + ignition shape |
| **HFT4** | 100 Hz | Pc, Ptank, Thrust (LC) | ~51 bar | Calibration + ignition shape |
| **SMT0033** | 50 Hz | Pc, Ptank, Thrust (LC) | ~61 bar | Primary reference (closest to flight) |
| **Flight** | 2.45 Hz | Pc, Ptank | ~61 bar | Reconstruction target |
| **Fluctus** | 50 Hz | Alt, Speed, Accel | — | Trajectory validation |

---
*Author: J. Martos — Abu Dhabi, UAE*  
*Motor: N₂O/Paraffin hybrid, L-class*

---
## 0 · Environment Setup

In [ ]:
!pip install -q rocketpy numpy matplotlib pandas scipy

In [ ]:
import os, subprocess

# Clone repo if running in Colab
if not os.path.exists('data/Flight_Complete_Burn.csv'):
    if not os.path.exists('hybrid-rocket-trajectory'):
        subprocess.check_call(
            'git clone https://github.com/jmartos-br/hybrid-rocket-trajectory.git',
            shell=True,
        )
    os.chdir('hybrid-rocket-trajectory')
    subprocess.check_call('git checkout rocketpy-retimed-motor', shell=True)

print('CWD:', os.getcwd())
print('Data files:', sorted(os.listdir('data')))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import interpolate, stats
import math

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (13, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.0,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
})

# Color palette for consistency across all plots
COLORS = {
    'SMT0033': '#FFB300',   # amber
    'HFT1':    '#7986CB',   # light blue
    'HFT3':    '#4DB6AC',   # teal
    'HFT4':    '#9E9E9E',   # gray
    'Flight':  '#E53935',   # red
    'Fluctus': '#AB47BC',   # purple
}

print('Environment ready.')

---
## 1 · Load All Data Sources

We load all six datasets into a unified structure. Each ground test has time-aligned thrust (from a load cell), chamber pressure (Pc), and tank pressure (Ptank). The flight only has Pc and Ptank at 2.45 Hz.

In [ ]:
# ── HFT1, HFT3, HFT4 (100 Hz, already processed) ──
hft1 = pd.read_csv('data/HFT1_processed.csv')
hft3 = pd.read_csv('data/HFT3_processed.csv')
hft4 = pd.read_csv('data/HFT4_processed.csv')

# ── SMT0033 (50 Hz, raw in daN) ──
smt_raw = pd.read_csv('data/SMT0033_extracted.csv')
smt = smt_raw.copy()
smt['time_s'] = smt['time_ms'] / 1000.0

# Align ignition: t = 0 when Pc first exceeds 2 bar
t_ign_smt = smt.loc[smt['chamber_pressure_bar'] > 2.0, 'time_s'].iloc[0]
smt['time_s'] -= t_ign_smt

# Convert thrust: subtract pre-ignition baseline, daN → N
pre_smt = smt['time_s'] < 0
thrust_baseline_smt = smt.loc[pre_smt, 'thrust_daN'].median()
smt['thrust_N'] = (smt['thrust_daN'] - thrust_baseline_smt) * 10.0

# ── Flight data (2.45 Hz, already retimed) ──
flight_raw = pd.read_csv('data/Flight_Complete_Burn.csv', sep=';')

# ── Fluctus telemetry (50 Hz) ──
fluctus_raw = pd.read_csv('data/fluctus_trimmed.csv')
fluctus = fluctus_raw.copy()
fluctus['time_s'] = fluctus['time_ms'] / 1000.0

# Align Fluctus: t=0 at launch (first positive altitude or acceleration spike)
launch_idx = fluctus.loc[fluctus['accel_ms2'] > 5.0].index[0]
t_launch = fluctus.loc[launch_idx, 'time_s']
fluctus['time_s'] -= t_launch

# ── Convenience dicts for iteration ──
# test_data: ground tests only (all have thrust load cell)
test_data = {
    'HFT1': hft1, 'HFT3': hft3, 'HFT4': hft4, 'SMT0033': smt,
}

# datasets: all sources including flight
datasets = {
    'HFT1': hft1, 'HFT3': hft3, 'HFT4': hft4,
    'SMT0033': smt, 'Flight': flight_raw,
}

print(f"{'Dataset':<10} {'Rows':>7} {'Rate':>8} {'Duration':>10}")
print('-' * 40)
for name, df in datasets.items():
    t_col = 'time_s' if 'time_s' in df.columns else 'time_real_s'
    t = df[t_col]
    n = len(df)
    dt = t.max() - t.min()
    rate = (n - 1) / dt if dt > 0 else 0
    print(f"{name:<10} {n:>7} {rate:>7.1f}Hz {dt:>9.2f} s")

print(f"{'Fluctus':<10} {len(fluctus):>7} {50:>7}Hz {fluctus['time_s'].max()-fluctus['time_s'].min():>9.2f} s")

---
## 2 · Exploratory Data Analysis — Ground Tests

### 2.1 · Overview of All Ground Tests

We plot thrust, chamber pressure, and tank pressure for all four ground tests overlaid. This reveals:
- **SMT0033** had the highest initial tank pressure (~61 bar), matching flight conditions
- **HFT1** shows the highest peak Pc (~38 bar) — likely due to different injector or grain geometry
- **HFT3 and HFT4** show moderate peaks (~33 bar), closer to flight measurements
- All tests show the characteristic **regressive thrust profile** of a blowdown hybrid

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

for name, df in test_data.items():
    t = df['time_s']
    mask = (t > -0.5) & (t < 28)
    c = COLORS[name]
    lw = 1.5 if name == 'SMT0033' else 0.8
    alpha = 0.9 if name == 'SMT0033' else 0.7

    axes[0].plot(t[mask], df.loc[mask, 'thrust_N'], color=c, lw=lw, alpha=alpha, label=name)
    axes[1].plot(t[mask], df.loc[mask, 'chamber_pressure_bar'], color=c, lw=lw, alpha=alpha, label=name)
    axes[2].plot(t[mask], df.loc[mask, 'tank_pressure_bar'], color=c, lw=lw, alpha=alpha, label=name)

axes[0].set_ylabel('Thrust [N]')
axes[0].set_title('All Ground Tests — Thrust, Chamber Pressure, Tank Pressure')
axes[0].legend(ncol=4, loc='upper right')

axes[1].set_ylabel('Chamber Pressure [bar]')
axes[1].legend(ncol=4, loc='upper right')

axes[2].set_ylabel('Tank Pressure [bar]')
axes[2].set_xlabel('Time from Ignition [s]')
axes[2].legend(ncol=4, loc='upper right')

plt.tight_layout()
plt.savefig('01_all_ground_tests_overview.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 · Key Performance Metrics — Ground Tests

Extract peak thrust, total impulse, burn duration, average thrust, and initial tank pressure for each test. This table quantifies the test-to-test variability.

In [ ]:
def compute_metrics(df, name, thrust_col='thrust_N', time_col='time_s',
                    pc_col='chamber_pressure_bar', ptank_col='tank_pressure_bar'):
    """Compute motor performance metrics for a ground test."""
    # Burn mask: thrust > 5 N
    burn = df[df[thrust_col] > 5.0].copy()
    if len(burn) == 0:
        return None
    
    t = burn[time_col].values
    f = burn[thrust_col].values
    
    total_impulse = float(np.trapz(f, t))
    burn_duration = t[-1] - t[0]
    avg_thrust = total_impulse / burn_duration if burn_duration > 0 else 0
    peak_thrust = f.max()
    peak_pc = burn[pc_col].max()
    
    # Initial tank pressure (median of pre-ignition)
    pre = df[df[time_col] < 0]
    ptank0 = pre[ptank_col].median() if len(pre) > 0 else df[ptank_col].iloc[0]
    
    # Motor class
    letter_idx = int(math.log2(total_impulse / 2.5))
    motor_letter = chr(ord('A') + letter_idx)
    
    return {
        'Test': name,
        'Ptank₀ [bar]': f'{ptank0:.1f}',
        'Peak Pc [bar]': f'{peak_pc:.1f}',
        'Peak Thrust [N]': f'{peak_thrust:.0f}',
        'Avg Thrust [N]': f'{avg_thrust:.0f}',
        'Total Impulse [Ns]': f'{total_impulse:.0f}',
        'Burn Time [s]': f'{burn_duration:.1f}',
        'Class': f'{motor_letter}{int(avg_thrust)}',
    }

metrics = []
for name, df in test_data.items():
    m = compute_metrics(df, name)
    if m:
        metrics.append(m)

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

### 2.3 · Ignition Transient Comparison (0–1 s)

The ignition transient is critical because the flight logger at 2.45 Hz **missed the first ~400 ms**. We examine the high-resolution (100 Hz) ignition profiles from HFT1, HFT3, and HFT4 to understand the shape we need to reconstruct.

Key observations:
- HFT1 shows a higher peak (~38 bar Pc) — different operating conditions
- HFT3 and HFT4 peak at ~33 bar, closely matching the flight's first measured point (33.7 bar at t=0.408 s)
- The ignition transient lasts ~200–400 ms before settling into steady burn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, df in test_data.items():
    mask = (df['time_s'] > -0.1) & (df['time_s'] < 1.0)
    c = COLORS[name]
    lw = 1.5 if name == 'SMT0033' else 1.0
    
    axes[0].plot(df.loc[mask, 'time_s'], df.loc[mask, 'chamber_pressure_bar'],
                 color=c, lw=lw, label=name)
    axes[1].plot(df.loc[mask, 'time_s'], df.loc[mask, 'thrust_N'],
                 color=c, lw=lw, label=name)

# Mark flight's first sample
flight_burn = flight_raw[flight_raw['pc_bar'] > 2.0]
axes[0].axvline(0.408, color=COLORS['Flight'], ls='--', alpha=0.7, label='Flight 1st sample')
axes[0].scatter([0.408], [33.7], color=COLORS['Flight'], s=80, zorder=5, marker='*')

axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Chamber Pressure [bar]')
axes[0].set_title('Ignition Transient — Chamber Pressure')
axes[0].legend()

axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('Thrust [N]')
axes[1].set_title('Ignition Transient — Thrust')
axes[1].legend()

plt.tight_layout()
plt.savefig('02_ignition_transient_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 · Pc → Thrust Calibration

### 3.1 · Physical Basis

For a rocket nozzle, thrust relates to chamber pressure via:

$$F = C_F \cdot A_t \cdot (P_c - P_{amb})$$

where $C_F$ is the thrust coefficient and $A_t$ is the throat area. For a given nozzle geometry, this simplifies to a linear relationship:

$$F = a \cdot P_{c,gauge} + b$$

We calibrate this using **all four ground tests** (which have load cells measuring thrust directly), fitting on the **steady-burn region** only (t > 0.5 s, Pc_gauge > 1 bar) to avoid the noisy ignition transient.

### 3.2 · Individual Test Calibrations

First, we examine the Pc → Thrust relationship for each test independently to check for consistency.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

fit_results = {}

for idx, (name, df) in enumerate(test_data.items()):
    ax = axes[idx]

    # Compute Pc baseline (atmospheric offset)
    # Some datasets have pre-ignition data (t < 0), others start at t=0
    pre = df[df['time_s'] < 0]
    if len(pre) > 10:
        # Use pre-ignition median (best estimate)
        pc0 = pre['chamber_pressure_bar'].median()
    else:
        # No pre-ignition data — estimate from minimum Pc in first 2s
        # (the sensor reads ~atmospheric before/between pressure transients)
        early = df[df['time_s'] < 2.0]['chamber_pressure_bar']
        pc0 = early.min()  # atmospheric baseline

    pc_gauge = np.maximum(df['chamber_pressure_bar'] - pc0, 0.0)

    # Steady-burn mask: after ignition transient, meaningful Pc and thrust
    cal_mask = (df['time_s'] > 0.5) & (pc_gauge > 1.0) & (df['thrust_N'] > 5.0)
    pcg = pc_gauge[cal_mask].values
    thr = df.loc[cal_mask, 'thrust_N'].values

    if len(pcg) < 10:
        print(f'  WARNING: {name} has only {len(pcg)} calibration points — skipping fit')
        fit_results[name] = {'a': 0, 'b': 0, 'r2': 0, 'pc0': pc0, 'n_points': len(pcg)}
        continue

    # Linear fit with intercept
    A = np.vstack([pcg, np.ones_like(pcg)]).T
    (a, b), *_ = np.linalg.lstsq(A, thr, rcond=None)
    a, b = float(a), float(b)

    # R²
    yhat = a * pcg + b
    ss_res = np.sum((thr - yhat)**2)
    ss_tot = np.sum((thr - np.mean(thr))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    fit_results[name] = {'a': a, 'b': b, 'r2': r2, 'pc0': pc0, 'n_points': len(pcg)}

    # Plot
    ax.scatter(pcg, thr, s=2, alpha=0.3, color=COLORS[name])
    x_fit = np.linspace(0, pcg.max() * 1.05, 100)
    ax.plot(x_fit, a * x_fit + b, 'k-', lw=2)
    ax.set_title(f'{name}: F = {a:.2f}·Pc + ({b:.1f})  [R²={r2:.4f}]')
    ax.set_xlabel('Pc_gauge [bar]')
    ax.set_ylabel('Thrust [N]')

plt.tight_layout()
plt.savefig('03_individual_calibrations.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'Test':<10} {'Slope (a)':>10} {'Intercept (b)':>14} {'R²':>8} {'Pc₀ [bar]':>10} {'N pts':>7}")
print('-' * 62)
for name, r in fit_results.items():
    print(f"{name:<10} {r['a']:>10.3f} {r['b']:>14.3f} {r['r2']:>8.5f} {r['pc0']:>10.3f} {r['n_points']:>7}")

### 3.3 · Combined Calibration — All Tests

We now pool all steady-burn data from all four tests to obtain the best overall Pc → Thrust transfer function. This uses the full statistical power of all available data.

We compare:
- **Origin fit:** $F = a \cdot Pc_{gauge}$ (physically motivated: zero thrust at zero pressure)
- **Intercept fit:** $F = a \cdot Pc_{gauge} + b$ (better R², captures real nozzle offset)

In [ ]:
# Pool all steady-burn calibration data
all_pcg = []
all_thr = []
all_labels = []

for name, df in test_data.items():
    pc0 = fit_results[name]['pc0']
    pc_gauge = np.maximum(df['chamber_pressure_bar'] - pc0, 0.0)
    cal_mask = (df['time_s'] > 0.5) & (pc_gauge > 1.0) & (df['thrust_N'] > 5.0)
    all_pcg.append(pc_gauge[cal_mask].values)
    all_thr.append(df.loc[cal_mask, 'thrust_N'].values)
    all_labels.extend([name] * cal_mask.sum())

pcg_all = np.concatenate(all_pcg)
thr_all = np.concatenate(all_thr)

# ── Origin fit: F = a · Pc_gauge ──
a_origin = float(np.dot(pcg_all, thr_all) / np.dot(pcg_all, pcg_all))
yhat_o = a_origin * pcg_all
ss_res_o = np.sum((thr_all - yhat_o)**2)
ss_tot = np.sum((thr_all - np.mean(thr_all))**2)
r2_origin = 1 - ss_res_o / ss_tot

# ── Intercept fit: F = a · Pc_gauge + b ──
A = np.vstack([pcg_all, np.ones_like(pcg_all)]).T
(a_combined, b_combined), *_ = np.linalg.lstsq(A, thr_all, rcond=None)
a_combined, b_combined = float(a_combined), float(b_combined)
yhat_i = a_combined * pcg_all + b_combined
ss_res_i = np.sum((thr_all - yhat_i)**2)
r2_combined = 1 - ss_res_i / ss_tot

# Residual statistics
residuals = thr_all - yhat_i
rmse = np.sqrt(np.mean(residuals**2))
mae = np.mean(np.abs(residuals))

print('='*60)
print('COMBINED Pc → THRUST CALIBRATION (ALL TESTS)')
print('='*60)
print(f'  Origin fit:    F = {a_origin:.3f} · Pc_gauge            (R² = {r2_origin:.5f})')
print(f'  Intercept fit: F = {a_combined:.3f} · Pc_gauge + ({b_combined:.3f})  (R² = {r2_combined:.5f})')
print(f'  RMSE: {rmse:.2f} N  |  MAE: {mae:.2f} N')
print(f'  Total calibration points: {len(pcg_all)}')
print(f'\n→ Using intercept fit for reconstruction.')

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter by test
offset = 0
for name in test_data:
    n = fit_results[name]['n_points']
    ax1.scatter(pcg_all[offset:offset+n], thr_all[offset:offset+n],
               s=3, alpha=0.3, color=COLORS[name], label=name)
    offset += n

x = np.linspace(0, pcg_all.max() * 1.05, 200)
ax1.plot(x, a_origin * x, 'r-', lw=2,
         label=f'Origin: F = {a_origin:.2f}·Pc  (R²={r2_origin:.4f})')
ax1.plot(x, a_combined * x + b_combined, 'k--', lw=2,
         label=f'Combined: F = {a_combined:.2f}·Pc + {b_combined:.1f}  (R²={r2_combined:.4f})')
ax1.set_xlabel('Pc_gauge [bar]')
ax1.set_ylabel('Thrust [N]')
ax1.set_title('Combined Pc → Thrust Calibration')
ax1.legend(fontsize=8)

# Residuals
ax2.hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(0, color='k', ls='--')
ax2.set_xlabel('Residual [N]')
ax2.set_ylabel('Count')
ax2.set_title(f'Calibration Residuals (RMSE = {rmse:.1f} N, MAE = {mae:.1f} N)')

plt.tight_layout()
plt.savefig('04_combined_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 · Thrust Coefficient Analysis

We can extract the effective thrust coefficient $C_F \cdot A_t$ from the slope of the calibration. For a conical nozzle with known throat area, this gives us $C_F$ — a key nozzle performance parameter.

$$C_F \cdot A_t = \text{slope} = a \quad [\text{N/bar}] \cdot \frac{1}{10^5} \quad [\text{m}^2]$$

In [ ]:
# Cf*At from slope
# slope a is in N/bar, convert: 1 bar = 1e5 Pa
# F = a * Pc_gauge [bar] → F = a * Pc_gauge * 1e5 [Pa] / 1e5
# So Cf*At = a / 1e5 [m²] → but a is already N/bar
# F [N] = Cf * At [m²] * Pc [Pa]
# a [N/bar] = Cf * At [m²] * 1e5 [Pa/bar]
# Cf * At = a / 1e5 [m²]

Cf_At = a_combined / 1e5  # m²

# Assumed throat diameter (from motor design)
d_throat = 0.025  # m (25 mm)
At = np.pi * (d_throat / 2)**2
Cf_eff = Cf_At / At

print(f'Effective Cf·At = {Cf_At*1e4:.4f} cm² = {Cf_At*1e6:.2f} mm²')
print(f'Assumed throat diameter: {d_throat*1000:.1f} mm')
print(f'Throat area At = {At*1e4:.4f} cm²')
print(f'Effective Cf = {Cf_eff:.3f}')
print(f'  (Theoretical ideal Cf for γ=1.2, Pe/Pc→0: ~1.5–1.7)')
print(f'  (Typical real nozzle Cf: 1.1–1.5)')

---
## 4 · Flight Data Overview

### 4.1 · Onboard Pc and Ptank

The flight computer logged chamber pressure and tank pressure at 2.45 Hz (every ~408 ms). The time axis has been corrected using Ptank blowdown correlation with SMT0033.

**Critical observation:** The jump from t=0 (Pc ≈ 0.6 bar) to t=0.408 s (Pc ≈ 33.7 bar) means the entire ignition transient was captured in a single sample interval. The true peak Pc occurred somewhere in this 408 ms window and was **not recorded**.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

t = flight_raw['time_real_s']

# Color by source type
is_burn = flight_raw['pc_bar'] > 2.0
pre_flight = flight_raw[~is_burn & (t < 1)]
burn_flight = flight_raw[is_burn]
post_flight = flight_raw[~is_burn & (t > 1)]

# Chamber Pressure
axes[0].plot(pre_flight['time_real_s'], pre_flight['pc_bar'], 'b.', ms=4, alpha=0.4)
axes[0].plot(burn_flight['time_real_s'], burn_flight['pc_bar'], 'r.-', ms=6, lw=1.2, label='Burn')
axes[0].plot(post_flight['time_real_s'], post_flight['pc_bar'], 'b.', ms=4, alpha=0.4)
axes[0].axhline(2.0, color='gray', ls=':', alpha=0.5, label='Pc = 2 bar threshold')
axes[0].annotate('First burn sample\nt=0.408s, Pc=33.7 bar',
                 xy=(0.408, 33.7), xytext=(3, 38),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=10, color='red')
axes[0].set_ylabel('Chamber Pressure [bar]')
axes[0].set_title('Flight Onboard Data (2.45 Hz, retimed)')
axes[0].legend()

# Tank Pressure
axes[1].plot(pre_flight['time_real_s'], pre_flight['ptank_bar'], 'g.', ms=4, alpha=0.4)
axes[1].plot(burn_flight['time_real_s'], burn_flight['ptank_bar'], 'g.-', ms=6, lw=1.2)
axes[1].plot(post_flight['time_real_s'], post_flight['ptank_bar'], 'g.', ms=4, alpha=0.4)
axes[1].set_ylabel('Tank Pressure [bar]')
axes[1].set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig('05_flight_onboard_data.png', dpi=150, bbox_inches='tight')
plt.show()

# Key statistics
pc0_flight = flight_raw.loc[flight_raw['time_real_s'] < 0, 'pc_bar'].median()
ptank0_flight = flight_raw.loc[flight_raw['time_real_s'] < 0, 'ptank_bar'].median()

print(f'Pre-ignition baselines:')
print(f'  Pc₀ = {pc0_flight:.3f} bar')
print(f'  Ptank₀ = {ptank0_flight:.2f} bar')
print(f'  (SMT0033 Ptank₀ = {smt.loc[smt["time_s"] < 0, "tank_pressure_bar"].median():.2f} bar)')
print(f'\nBurn samples: {is_burn.sum()}')
print(f'First burn: t={burn_flight["time_real_s"].iloc[0]:.3f}s, Pc={burn_flight["pc_bar"].iloc[0]:.1f} bar')
print(f'Last burn:  t={burn_flight["time_real_s"].iloc[-1]:.3f}s, Pc={burn_flight["pc_bar"].iloc[-1]:.2f} bar')

### 4.2 · Flight vs SMT0033 — Ptank Blowdown Comparison

Since the flight and SMT0033 had nearly identical initial tank pressures (~61 bar), we compare their Ptank depletion curves. A close match confirms they operated under similar conditions and validates using SMT0033 as the primary reference.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Ptank vs Time
smt_mask = (smt['time_s'] > -1) & (smt['time_s'] < 30)
ax1.plot(smt.loc[smt_mask, 'time_s'], smt.loc[smt_mask, 'tank_pressure_bar'],
         color=COLORS['SMT0033'], lw=1.5, label='SMT0033')
ax1.plot(burn_flight['time_real_s'], burn_flight['ptank_bar'],
         'o-', color=COLORS['Flight'], ms=4, lw=1, label='Flight')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Tank Pressure [bar]')
ax1.set_title('Ptank Blowdown — Flight vs SMT0033')
ax1.legend()

# Pc vs Ptank (operating curve)
smt_bm = (smt['time_s'] > 0.5) & (smt['thrust_N'] > 5)
ax2.plot(smt.loc[smt_bm, 'tank_pressure_bar'], smt.loc[smt_bm, 'chamber_pressure_bar'],
         color=COLORS['SMT0033'], lw=1, alpha=0.7, label='SMT0033')
ax2.plot(burn_flight['ptank_bar'], burn_flight['pc_bar'],
         'o-', color=COLORS['Flight'], ms=5, label='Flight')
ax2.set_xlabel('Tank Pressure [bar]')
ax2.set_ylabel('Chamber Pressure [bar]')
ax2.set_title('Pc vs Ptank (operating curve)')
ax2.legend()
ax2.invert_xaxis()

plt.tight_layout()
plt.savefig('06_flight_vs_smt0033_blowdown.png', dpi=150, bbox_inches='tight')
plt.show()

# Pc/Ptank ratio
ratio_flight = burn_flight['pc_bar'].values / burn_flight['ptank_bar'].values
ratio_smt = (smt.loc[smt_bm, 'chamber_pressure_bar'].values /
             smt.loc[smt_bm, 'tank_pressure_bar'].values)
print(f'Pc/Ptank ratio — Flight: {np.median(ratio_flight):.3f} (median)')
print(f'Pc/Ptank ratio — SMT0033: {np.median(ratio_smt):.3f} (median)')

---
## 5 · Flight Thrust Reconstruction — Per-Phase Approach

We reconstruct the flight thrust curve in three phases, using the best available data for each:

| Phase | Time Window | Method | Data Source |
|-------|------------|--------|------------|
| **1 — Ignition** | 0 → 0.408 s | Shape interpolation | HFT3 + HFT4 (100 Hz) |
| **2 — Steady burn** | 0.408 → 23.3 s | Pc→Thrust calibration | Flight Pc + combined fit |
| **3 — Shutdown** | 23.3 → 23.7 s | Linear ramp to zero | Last flight Pc sample |

### 5.1 · Phase 1 — Ignition Transient Reconstruction

The flight logger missed the ignition transient (0–0.408 s). We reconstruct it by:
1. Extracting the ignition Pc profile from HFT3 and HFT4 (which peaked at ~33 bar, similar to flight)
2. Averaging and normalizing these profiles
3. Scaling to match the flight's boundary condition: Pc = 33.7 bar at t = 0.408 s

In [ ]:
# ── Extract ignition profiles from HFT3 and HFT4 ──
# These tests have 100 Hz data and peaked at ~33 bar Pc, similar to flight

def extract_ignition_profile(df, name, t_end=0.5):
    """Extract the ignition thrust profile from 0 to t_end seconds."""
    mask = (df['time_s'] >= 0) & (df['time_s'] <= t_end)
    t = df.loc[mask, 'time_s'].values
    pc = df.loc[mask, 'chamber_pressure_bar'].values
    thrust = df.loc[mask, 'thrust_N'].values
    return t, pc, thrust

t_ign3, pc_ign3, f_ign3 = extract_ignition_profile(hft3, 'HFT3', t_end=0.5)
t_ign4, pc_ign4, f_ign4 = extract_ignition_profile(hft4, 'HFT4', t_end=0.5)

# Interpolate both onto a common time grid (100 Hz)
t_common = np.arange(0, 0.41, 0.01)  # 0 to 0.40s at 100 Hz

interp3_pc = np.interp(t_common, t_ign3, pc_ign3)
interp4_pc = np.interp(t_common, t_ign4, pc_ign4)
interp3_f = np.interp(t_common, t_ign3, f_ign3)
interp4_f = np.interp(t_common, t_ign4, f_ign4)

# Work in GAUGE pressure space for physical consistency
# Each HFT has its own Pc baseline — subtract before averaging and scaling
pc0_hft3 = fit_results['HFT3']['pc0']
pc0_hft4 = fit_results['HFT4']['pc0']
interp3_pcg = np.maximum(interp3_pc - pc0_hft3, 0)
interp4_pcg = np.maximum(interp4_pc - pc0_hft4, 0)
avg_pcg_ign = (interp3_pcg + interp4_pcg) / 2

# Scale gauge Pc so that value at t=0.408 matches flight's gauge Pc
pc_flight_boundary = 33.7  # flight's first burn sample Pc [bar]
flight_pcg_boundary = pc_flight_boundary - pc0_flight  # flight gauge Pc
pcg_at_boundary = avg_pcg_ign[-1]
scale_factor_pc = flight_pcg_boundary / pcg_at_boundary if pcg_at_boundary > 0 else 1.0

scaled_pcg_ign = avg_pcg_ign * scale_factor_pc
scaled_pc_ign = scaled_pcg_ign + pc0_flight  # absolute Pc for plotting

# Convert gauge Pc directly to thrust using the combined calibration
scaled_f_ign = np.maximum(a_combined * scaled_pcg_ign + b_combined, 0)

print(f'HFT3 Pc0 baseline: {pc0_hft3:.3f} bar')
print(f'HFT4 Pc0 baseline: {pc0_hft4:.3f} bar')
print(f'Flight Pc0 baseline: {pc0_flight:.3f} bar')
print(f'Average Pc_gauge at t=0.40s: {pcg_at_boundary:.1f} bar')
print(f'Flight Pc_gauge at t=0.408s: {flight_pcg_boundary:.1f} bar')
print(f'Scale factor: {scale_factor_pc:.3f}')
print(f'\nReconstructed ignition peak Pc: {scaled_pc_ign.max():.1f} bar')
print(f'Reconstructed ignition peak thrust: {scaled_f_ign.max():.0f} N')

In [ ]:
# Visualize the ignition reconstruction
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(t_ign3, pc_ign3, color=COLORS['HFT3'], alpha=0.5, label='HFT3 (raw)')
ax1.plot(t_ign4, pc_ign4, color=COLORS['HFT4'], alpha=0.5, label='HFT4 (raw)')
ax1.plot(t_common, scaled_pc_ign, 'r-', lw=2, label='Reconstructed (scaled avg)')
ax1.scatter([0.408], [33.7], color=COLORS['Flight'], s=100, zorder=5, marker='*',
            label=f'Flight 1st sample')
ax1.axvspan(0, 0.408, alpha=0.08, color='red', label='Missed window')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Chamber Pressure [bar]')
ax1.set_title('Phase 1 — Ignition Pc Reconstruction')
ax1.legend(fontsize=9)

ax2.plot(t_ign3, f_ign3, color=COLORS['HFT3'], alpha=0.5, label='HFT3 (raw)')
ax2.plot(t_ign4, f_ign4, color=COLORS['HFT4'], alpha=0.5, label='HFT4 (raw)')
ax2.plot(t_common, scaled_f_ign, 'r-', lw=2, label='Reconstructed thrust')
ax2.axvspan(0, 0.408, alpha=0.08, color='red')
ax2.set_xlabel('Time [s]')
ax2.set_ylabel('Thrust [N]')
ax2.set_title('Phase 1 — Ignition Thrust Reconstruction')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('07_ignition_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 · Phase 2 — Steady Burn Reconstruction

For the measured portion (57 burn samples at 2.45 Hz), we apply the combined Pc→Thrust calibration directly.

In [ ]:
# Apply calibration to flight burn data
flight = flight_raw.copy()
flight['pc_gauge'] = np.maximum(flight['pc_bar'] - pc0_flight, 0.0)
flight['thrust_N'] = a_combined * flight['pc_gauge'] + b_combined

# Zero out pre-ignition and clamp negatives
flight.loc[flight['pc_bar'] < 2.0, 'thrust_N'] = 0.0
flight['thrust_N'] = np.maximum(flight['thrust_N'], 0.0)

# Burn region only
burn = flight[flight['thrust_N'] > 0].copy()

print(f'Calibration model: F = {a_combined:.3f} · Pc_gauge + ({b_combined:.3f})')
print(f'Pc baseline (flight): {pc0_flight:.3f} bar')
print(f'\nBurn samples: {len(burn)}')
print(f'Time range: {burn["time_real_s"].iloc[0]:.3f} – {burn["time_real_s"].iloc[-1]:.3f} s')
print(f'Thrust range: {burn["thrust_N"].min():.1f} – {burn["thrust_N"].max():.1f} N')

### 5.3 · Assemble Complete Thrust Curve

Combine all three phases:
1. **Phase 1 (0–0.408s):** Reconstructed ignition from HFT3/HFT4
2. **Phase 2 (0.408–23.3s):** Calibrated flight Pc → Thrust
3. **Phase 3 (shutdown):** Linear ramp to zero after last burn sample

In [ ]:
# ── Assemble the complete thrust curve ──

# Phase 1: ignition (exclude the last point to avoid overlap with phase 2)
t_phase1 = t_common[:-1]  # 0 to 0.39s
f_phase1 = scaled_f_ign[:-1]

# Phase 2: flight burn data
t_phase2 = burn['time_real_s'].values
f_phase2 = burn['thrust_N'].values

# Phase 3: shutdown ramp (linear to zero in 0.4s after last sample)
t_last = t_phase2[-1]
f_last = f_phase2[-1]
t_phase3 = np.array([t_last + 0.2, t_last + 0.4])
f_phase3 = np.array([f_last * 0.5, 0.0])

# Concatenate all phases
t_full = np.concatenate([t_phase1, t_phase2, t_phase3])
f_full = np.concatenate([f_phase1, f_phase2, f_phase3])

# Compute metrics
total_impulse = float(np.trapz(f_full, t_full))
burn_duration = t_full[-1] - t_full[0]
avg_thrust = total_impulse / burn_duration
peak_thrust = f_full.max()
letter_idx = int(math.log2(total_impulse / 2.5))
motor_letter = chr(ord('A') + letter_idx)
designation = f'{motor_letter}{int(avg_thrust)}'

print('='*60)
print('   RECONSTRUCTED FLIGHT MOTOR — FULL CURVE')
print('='*60)
print(f'  Peak thrust:      {peak_thrust:>8.0f} N')
print(f'  Average thrust:   {avg_thrust:>8.0f} N')
print(f'  Total impulse:    {total_impulse:>8.0f} N·s')
print(f'  Burn duration:    {burn_duration:>8.1f} s')
print(f'  Motor class:      {designation}')
print(f'  Model:            F = {a_combined:.1f}·Pc + ({b_combined:.1f})')
print('='*60)

In [ ]:
# ── Master comparison plot: Flight reconstructed vs all references ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: Chamber Pressure
for name, df in test_data.items():
    if name == 'Flight':
        continue
    mask = (df['time_s'] > -0.5) & (df['time_s'] < 25)
    ax1.plot(df.loc[mask, 'time_s'], df.loc[mask, 'chamber_pressure_bar'],
             color=COLORS[name], lw=0.8, alpha=0.6, label=name)

# Flight Pc (measured points)
ax1.plot(burn['time_real_s'], burn['pc_bar'], 'o',
         color=COLORS['Flight'], ms=5, label='Flight (measured)', zorder=5)
# Reconstructed ignition Pc
ax1.plot(t_common, scaled_pc_ign, '--',
         color=COLORS['Flight'], lw=1.5, alpha=0.8, label='Flight (reconstructed ignition)')

ax1.set_ylabel('Chamber Pressure [bar]')
ax1.set_title('Flight Thrust Reconstruction — Pc and Thrust vs All References')
ax1.legend(ncol=3, fontsize=9)

# Bottom: Thrust
for name, df in test_data.items():
    if name == 'Flight':
        continue
    mask = (df['time_s'] > -0.5) & (df['time_s'] < 25)
    ax2.plot(df.loc[mask, 'time_s'], df.loc[mask, 'thrust_N'],
             color=COLORS[name], lw=0.8, alpha=0.6, label=name)

# Full reconstructed flight thrust
ax2.plot(t_full, f_full, 'o-', color=COLORS['Flight'], ms=4, lw=1.5,
         label='Flight RECONSTRUCTED', zorder=5)

# Phase annotation
ax2.axvspan(0, 0.408, alpha=0.08, color='red')
ax2.annotate('Phase 1\n(HFT3/4 shape)', xy=(0.15, peak_thrust * 0.95),
             fontsize=9, color='red', ha='center')

# Info box
info = (f'Reconstructed Flight Motor\n'
        f'Peak: {peak_thrust:.0f} N | Avg: {avg_thrust:.0f} N\n'
        f'Impulse: {total_impulse:.0f} Ns | Burn: {burn_duration:.1f} s\n'
        f'Class: {designation} | Model: F={a_combined:.1f}*Pc+({b_combined:.1f})')
ax2.text(0.35, 0.95, info, transform=ax2.transAxes, fontsize=9,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

ax2.set_ylabel('Thrust [N]')
ax2.set_xlabel('Time from Ignition [s]')
ax2.legend(ncol=3, fontsize=9)

plt.tight_layout()
plt.savefig('08_flight_thrust_reconstruction_full.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6 · Validation

### 6.1 · Cross-Check Against Independent Metrics

We validate the reconstructed thrust curve against multiple independent references.

In [ ]:
# SMT0033 reference metrics
smt_burn = smt[smt['thrust_N'] > 5]
smt_impulse = float(np.trapz(smt_burn['thrust_N'], smt_burn['time_s']))
smt_peak = smt_burn['thrust_N'].max()
smt_duration = smt_burn['time_s'].iloc[-1] - smt_burn['time_s'].iloc[0]
smt_avg = smt_impulse / smt_duration

# Without ignition reconstruction (for comparison)
impulse_no_ign = float(np.trapz(f_phase2, t_phase2))

print('='*70)
print('VALIDATION — RECONSTRUCTED FLIGHT vs REFERENCES')
print('='*70)
print(f"{'Metric':<25} {'Flight Recon':>14} {'SMT0033':>14} {'Flight (no ign)':>16}")
print('-'*70)
print(f"{'Peak Thrust [N]':<25} {peak_thrust:>14.0f} {smt_peak:>14.0f} {f_phase2.max():>16.0f}")
print(f"{'Avg Thrust [N]':<25} {avg_thrust:>14.0f} {smt_avg:>14.0f} {impulse_no_ign/(t_phase2[-1]-t_phase2[0]):>16.0f}")
print(f"{'Total Impulse [Ns]':<25} {total_impulse:>14.0f} {smt_impulse:>14.0f} {impulse_no_ign:>16.0f}")
print(f"{'Burn Duration [s]':<25} {burn_duration:>14.1f} {smt_duration:>14.1f} {t_phase2[-1]-t_phase2[0]:>16.1f}")
print(f"{'Motor Class':<25} {designation:>14} {'L'+str(int(smt_avg)):>14} {'—':>16}")
print('='*70)

### 6.2 · Thrust vs Ptank Blowdown Correlation

The thrust-vs-Ptank curve removes timing and shows the fundamental operating relationship. If the flight reconstruction is correct, it should overlap with SMT0033 (same motor, same Ptank₀).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Thrust vs Time comparison
for name, df in test_data.items():
    if name == 'Flight':
        continue
    mask = (df['time_s'] > -0.5) & (df['time_s'] < 25)
    ax1.plot(df.loc[mask, 'time_s'], df.loc[mask, 'thrust_N'],
             color=COLORS[name], lw=0.7, alpha=0.5, label=name)

ax1.plot(t_full, f_full, 'o-', color=COLORS['Flight'], ms=3, lw=1.2,
         label='Flight (reconstructed)')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Thrust [N]')
ax1.set_title('Thrust vs Time — All Tests')
ax1.legend(fontsize=9)

# Thrust vs Ptank
for name, df in test_data.items():
    if name == 'Flight':
        continue
    bm = (df['time_s'] > 0.5) & (df['thrust_N'] > 5)
    ds = df[bm].sort_values('tank_pressure_bar', ascending=False)
    ax2.plot(ds['tank_pressure_bar'], ds['thrust_N'],
             color=COLORS[name], lw=0.7, alpha=0.5, label=name)

burn_s = burn.sort_values('ptank_bar', ascending=False)
ax2.plot(burn_s['ptank_bar'], burn_s['thrust_N'],
         'o-', color=COLORS['Flight'], ms=4, lw=1.2, label='Flight (reconstructed)')
ax2.invert_xaxis()
ax2.set_xlabel('Tank Pressure [bar]  →  blowdown')
ax2.set_ylabel('Thrust [N]')
ax2.set_title('Thrust vs Ptank (blowdown correlation)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('09_validation_thrust_vs_ptank.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 · Calibration Uncertainty Envelope

We quantify the uncertainty in the reconstructed thrust by examining the spread across individual test calibrations. The envelope shows the range of possible thrust values at each flight Pc point.

In [ ]:
# Compute thrust using each individual test's calibration
t_burn = burn['time_real_s'].values
pc_gauge_burn = burn['pc_gauge'].values

thrust_per_test = {}
for name, r in fit_results.items():
    thrust_per_test[name] = np.maximum(r['a'] * pc_gauge_burn + r['b'], 0.0)

# Stack for envelope
all_thrust = np.column_stack(list(thrust_per_test.values()))
f_min = all_thrust.min(axis=1)
f_max = all_thrust.max(axis=1)
f_mean = all_thrust.mean(axis=1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(t_burn, f_min, f_max, alpha=0.2, color='red',
                label='Calibration spread (min\u2013max across tests)')
ax.plot(t_burn, f_phase2, 'ro-', ms=4, lw=1.2,
        label=f'Combined fit: F={a_combined:.1f}\u00b7Pc+({b_combined:.1f})')
ax.plot(t_burn, f_mean, 'k--', lw=0.8, alpha=0.5, label='Mean of individual fits')

ax.set_xlabel('Time [s]')
ax.set_ylabel('Thrust [N]')
ax.set_title('Reconstruction Uncertainty \u2014 Spread Across Individual Test Calibrations')
ax.legend()

plt.tight_layout()
plt.savefig('10_uncertainty_envelope.png', dpi=150, bbox_inches='tight')
plt.show()

# Uncertainty stats
spread = f_max - f_min
idx_peak_pc = np.argmax(pc_gauge_burn)
print(f'Thrust uncertainty (max\u2212min across calibrations):')
print(f'  Mean spread: {spread.mean():.1f} N ({spread.mean()/f_mean.mean()*100:.1f}%)')
print(f'  Max spread:  {spread.max():.1f} N')
print(f'  At peak Pc:  {spread[idx_peak_pc]:.1f} N (Pc_gauge = {pc_gauge_burn[idx_peak_pc]:.1f} bar)')

---
## 7 · Export `.eng` File

Export the reconstructed thrust curve in **RASP `.eng` format** — compatible with RocketPy, OpenRocket, and RASAero.

We export two versions:
1. **With ignition reconstruction** (full curve including Phase 1)
2. **Without ignition reconstruction** (starts at first flight sample)

In [ ]:
def write_eng_file(filepath, t_arr, f_arr, designation, comment='',
                   diameter_mm=100, length_mm=1330,
                   propellant_mass_kg=2.42, total_mass_kg=9.32):
    """Write a RASP .eng file."""
    # Shift t=0 to first point
    t = t_arr - t_arr[0]
    f = f_arr.copy()
    
    # Ensure ends at zero
    if f[-1] > 0:
        t = np.append(t, t[-1] + 0.01)
        f = np.append(f, 0.0)
    
    impulse = float(np.trapz(f, t))
    burn_time = float(t[-1])
    avg = impulse / burn_time
    
    with open(filepath, 'w') as fh:
        fh.write(f'; {comment}\n')
        fh.write(f'; Calibration: F = {a_combined:.3f} * Pc_gauge + ({b_combined:.3f})\n')
        fh.write(f'; Total Impulse: {impulse:.1f} Ns, Max Thrust: {f.max():.1f} N, '
                 f'Avg Thrust: {avg:.1f} N, Burn Time: {burn_time:.1f} s\n')
        fh.write(f'{designation} {diameter_mm} {length_mm} 0 '
                 f'{propellant_mass_kg:.3f} {total_mass_kg:.3f} FlightRecon\n')
        for ti, fi in zip(t, f):
            fh.write(f'  {ti:.4f}    {fi:.3f}\n')
    
    print(f'Wrote: {filepath}')
    print(f'  Designation: {designation}, Impulse: {impulse:.0f} Ns, '
          f'Peak: {f.max():.0f} N, Avg: {avg:.0f} N, Burn: {burn_time:.1f} s')
    return filepath

# Version 1: With ignition reconstruction
eng_full = write_eng_file(
    'data/Flight_Reconstructed_Full.eng',
    t_full, f_full, designation,
    comment='Flight Reconstructed (with HFT3/4 ignition synthesis)'
)
print()

# Version 2: Without ignition (starts at first flight sample)
t_noign = np.concatenate([t_phase2, t_phase3])
f_noign = np.concatenate([f_phase2, f_phase3])
impulse_noign = float(np.trapz(f_noign, t_noign))
avg_noign = impulse_noign / (t_noign[-1] - t_noign[0])
letter_noign = chr(ord('A') + int(math.log2(impulse_noign / 2.5)))
desig_noign = f'{letter_noign}{int(avg_noign)}'

eng_noign = write_eng_file(
    'data/Flight_Reconstructed_NoIgn.eng',
    t_noign, f_noign, desig_noign,
    comment='Flight Reconstructed (no ignition synthesis, starts at t=0.408s)'
)

---
## 8 · RocketPy Trajectory Simulation

### 8.1 · Simulation Setup

| Parameter | Value |
|-----------|-------|
| **Airframe** | |
| Diameter | 100 mm |
| Total length | ~2.6 m |
| Airframe dry mass | 3.78 kg |
| **Motor** | |
| Motor dry mass | 6.9 kg |
| Propellant mass | 2.42 kg |
| Chamber diameter | 100 mm |
| Chamber length | 1330 mm |
| Nozzle exit diameter | 50 mm |
| **Launch** | |
| Rail length | 7 m |
| Inclination | 83.5° |
| Heading | 90° (East) |
| **Recovery** | |
| Parachute Cd×S | 5.78 m² (main, at apogee) |

In [ ]:
from rocketpy import Environment, Rocket, Flight, GenericMotor

# ── Environment: launch site near Abu Dhabi, 13 Feb 2026 ──
env = Environment(latitude=24.18133, longitude=53.688379, elevation=5)
env.set_date((2026, 2, 13, 12))
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[(0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45),
            (1542, -0.62), (3164, -0.51), (5854, 12.69)],
    wind_v=[(0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46),
            (1542, 1.40), (3164, -0.51), (5854, 4.62)],
    pressure=[(0, 101500), (135, 100000), (818, 92500),
              (1542, 85000), (3164, 70000), (5854, 50000)],
    temperature=[(0, 302.95), (135, 301.65), (818, 295.25),
                 (1542, 288.75), (3164, 281.35), (5854, 264.25)],
)

print('Environment configured: Abu Dhabi launch site')
print(f'  Elevation: {env.elevation} m ASL')
print(f'  Date: 2026-02-13 12:00 UTC')

In [ ]:
# ── Motor (using full reconstruction with ignition) ──
# Read burn time directly from the .eng file for consistency
eng_times = []
with open(eng_full) as ef:
    for line in ef:
        line = line.strip()
        if not line or line.startswith(';'):
            continue
        parts = line.split()
        if len(parts) == 2:
            try:
                eng_times.append(float(parts[0]))
            except ValueError:
                pass
eng_burn_time = eng_times[-1] if eng_times else burn_duration

motor = GenericMotor(
    thrust_source=eng_full,
    burn_time=eng_burn_time,
    chamber_radius=0.05,
    chamber_height=1.33,
    chamber_position=1.33 / 2,
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33 / 2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

# ── Rocket ──
rocket = Rocket(
    radius=0.05,
    mass=3.780,
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='data/poweroff_drag.csv',
    power_on_drag='data/poweron_drag.csv',
    center_of_mass_without_motor=1.869,
    coordinate_system_orientation='tail_to_nose',
)
rocket.add_motor(motor, position=0.0)
rocket.add_nose(length=0.3, kind='ogive', position=2.600)
rocket.add_trapezoidal_fins(
    n=4, root_chord=0.145, tip_chord=0.065, span=0.08,
    sweep_length=0.11, cant_angle=0.5, position=0.145,
)
rocket.add_tail(
    top_radius=0.05, bottom_radius=0.03, length=0.055, position=0.0,
)
rocket.set_rail_buttons(
    upper_button_position=1.80, lower_button_position=0.40, angular_position=88,
)

# ── Parachute ──
def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * np.pi * (1.8288 / 2) ** 2,
    trigger=main_trigger,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

print('Rocket configured.')
print(f'  Motor burn time (from .eng): {eng_burn_time:.2f} s')

In [ ]:
# ── Run simulation ──
sim = Flight(
    rocket=rocket,
    environment=env,
    rail_length=7.0,
    inclination=83.5,
    heading=90,
    max_time=600,
    time_overshoot=True,
)

print('='*50)
print('  ROCKETPY SIMULATION RESULTS')
print('='*50)
print(f'  Apogee AGL:        {float(sim.apogee - env.elevation):>8.0f} m')
print(f'  Max speed:         {float(sim.max_speed):>8.1f} m/s')
print(f'  Max Mach:          {float(sim.max_mach_number):>8.2f}')
print(f'  Max acceleration:  {float(sim.max_acceleration):>8.1f} m/s²')
print(f'  Time of apogee:    {float(sim.apogee_time):>8.1f} s')
print(f'  Impact time:       {float(sim.t_final):>8.1f} s')
print('='*50)

### 8.2 · Simulation Detail

In [ ]:
sim.all_info()

---
## 9 · Trajectory Validation — Simulation vs Fluctus Telemetry

We compare the RocketPy simulation output with the Fluctus altimeter telemetry data recorded during the actual flight. This is the ultimate validation: does the reconstructed motor, when simulated, reproduce the observed trajectory?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Extract simulation data
sim_t = np.array(sim.altitude.source)[:, 0]
sim_alt = np.array(sim.altitude.source)[:, 1] - env.elevation  # AGL
sim_speed = np.array(sim.speed.source)
sim_accel = np.array(sim.acceleration.source)

# Fluctus data (trim to relevant range)
flu_mask = (fluctus['time_s'] > -5) & (fluctus['time_s'] < 200)
flu = fluctus[flu_mask]

# Altitude
axes[0].plot(sim_t, sim_alt, 'b-', lw=1.5, label='RocketPy simulation')
axes[0].plot(flu['time_s'], flu['altitude_m'], '--',
             color=COLORS['Fluctus'], lw=1.2, label='Fluctus telemetry')
axes[0].set_ylabel('Altitude AGL [m]')
axes[0].set_title('Trajectory Validation — Simulation vs Flight Telemetry')
axes[0].legend()

# Speed
axes[1].plot(sim_speed[:, 0], sim_speed[:, 1], 'b-', lw=1.5, label='RocketPy')
axes[1].plot(flu['time_s'], flu['speed_ms'], '--',
             color=COLORS['Fluctus'], lw=1.2, label='Fluctus')
axes[1].set_ylabel('Speed [m/s]')
axes[1].legend()

# Acceleration
axes[2].plot(sim_accel[:, 0], sim_accel[:, 1], 'b-', lw=1.5, alpha=0.7, label='RocketPy')
axes[2].plot(flu['time_s'], flu['accel_ms2'], '--',
             color=COLORS['Fluctus'], lw=1.0, alpha=0.7, label='Fluctus')
axes[2].set_ylabel('Acceleration [m/s²]')
axes[2].set_xlabel('Time [s]')
axes[2].set_xlim(-5, 150)
axes[2].legend()

plt.tight_layout()
plt.savefig('11_trajectory_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Apogee comparison
flu_apogee = flu['altitude_m'].max()
sim_apogee_agl = float(sim.apogee - env.elevation)
print(f'\nApogee comparison:')
print(f'  Fluctus telemetry: {flu_apogee:.0f} m AGL')
print(f'  RocketPy sim:      {sim_apogee_agl:.0f} m AGL')
print(f'  Difference:        {sim_apogee_agl - flu_apogee:+.0f} m ({(sim_apogee_agl - flu_apogee)/flu_apogee*100:+.1f}%)')

---
## 10 · Summary & Conclusions

### Key Results

In [ ]:
print('='*70)
print('  FLIGHT THRUST RECONSTRUCTION — SUMMARY')
print('='*70)
print()
print('Calibration:')
print(f'  Model:              F = {a_combined:.3f} · Pc_gauge + ({b_combined:.3f})')
print(f'  R²:                 {r2_combined:.5f}')
print(f'  RMSE:               {rmse:.1f} N')
print(f'  Calibration points: {len(pcg_all)} (pooled from HFT1, HFT3, HFT4, SMT0033)')
print()
print('Reconstructed Flight Motor:')
print(f'  Designation:        {designation}')
print(f'  Peak thrust:        {peak_thrust:.0f} N')
print(f'  Average thrust:     {avg_thrust:.0f} N')
print(f'  Total impulse:      {total_impulse:.0f} N·s')
print(f'  Burn duration:      {burn_duration:.1f} s')
print()
print('Reconstruction Method:')
print(f'  Phase 1 (0–0.4s):   Ignition transient from HFT3+HFT4 average, scaled')
print(f'  Phase 2 (0.4–23s):  Combined Pc→Thrust calibration on flight Pc')
print(f'  Phase 3 (23–23.7s): Linear shutdown ramp')
print()
print('RocketPy Simulation:')
print(f'  Simulated apogee:   {float(sim.apogee - env.elevation):.0f} m AGL')
print(f'  Max speed:          {float(sim.max_speed):.1f} m/s (Mach {float(sim.max_mach_number):.2f})')
print(f'  Max acceleration:   {float(sim.max_acceleration):.1f} m/s²')
print('='*70)

---
## 11 · Export & Download (Colab)

Zip all generated files for download.

In [ ]:
import glob, zipfile, shutil

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

out_dir = 'outputs_full_analysis'
os.makedirs(out_dir, exist_ok=True)

# Copy .eng files and plots
for p in glob.glob('data/Flight_Reconstructed*.eng') + glob.glob('*.png'):
    try:
        shutil.copy2(p, os.path.join(out_dir, os.path.basename(p)))
    except Exception:
        pass

# Save reconstructed thrust data as CSV
recon_df = pd.DataFrame({'time_s': t_full, 'thrust_N': f_full})
recon_csv = os.path.join(out_dir, 'flight_reconstructed_thrust.csv')
recon_df.to_csv(recon_csv, index=False)
print(f'Saved: {recon_csv}')

# Zip
zip_path = 'flight_analysis_complete.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk(out_dir):
        for fn in fnames:
            full = os.path.join(root, fn)
            z.write(full, arcname=os.path.relpath(full, '.'))

print(f'\nWrote {zip_path}')
if IN_COLAB:
    colab_files.download(zip_path)
else:
    print('(Not in Colab — download manually)')